# 6 - Transactions, KV, and Data Modeling

This notebook focuses on the database side of SochDB.

A lot of first-touch demos emphasize retrieval. This walkthrough makes the embedded data layer real by showing:

- basic key-value storage
- ACID transactions
- rollback behavior
- batch and prefix operations
- path-style organization
- SQL inside transactional workflows

If you want to understand SochDB as more than just a retrieval demo, this is one of the best notebooks to run.

### Step 0: Install Packages

Recommended install:

```bash
pip install sochdb
```

In [6]:
!pip install sochdb

import json
import shutil
import time
from pathlib import Path

### Step 1: Initialize Database
Open an embedded local database for the transaction and KV examples.

In [7]:
from sochdb import Database

DB_PATH = Path("./transactions_demo_db")
if DB_PATH.exists():
    shutil.rmtree(DB_PATH)

db = Database.open(str(DB_PATH))
print(f"Database opened for transaction demos at {DB_PATH}.")

Database opened for transaction demos at transactions_demo_db.


### Step 2: Basic Key-Value Operations
At the core, SochDB can be used as a local embedded KV store. Start with simple put/get/exists operations.

In [8]:
# Simple KV operations
db.put(b"user:alice:balance", b"1000")
db.put(b"user:bob:balance", b"500")

alice_balance = db.get(b"user:alice:balance")
bob_balance = db.get(b"user:bob:balance")

print(f"Alice's balance: {alice_balance.decode()}")
print(f"Bob's balance: {bob_balance.decode()}")
print(f"Key exists? {db.get(b'user:alice:balance') is not None}")

Alice's balance: 1000
Bob's balance: 500
Key exists? True


### Step 3: Atomic Transactions (Context Manager)
Transfer 200 from Alice to Bob **atomically**. Either both the debit and credit happen, or neither does.

In [9]:
# Atomic fund transfer using context manager
with db.transaction() as txn:
    # Read current balances through the transaction object.
    alice_bal = int(txn.get(b"user:alice:balance").decode())
    bob_bal = int(txn.get(b"user:bob:balance").decode())
    
    transfer_amount = 200
    
    # Debit Alice, Credit Bob
    txn.put(b"user:alice:balance", str(alice_bal - transfer_amount).encode())
    txn.put(b"user:bob:balance", str(bob_bal + transfer_amount).encode())
    
    # Transaction commits automatically when exiting the `with` block

# Verify
print(f"Alice's balance after transfer: {db.get(b'user:alice:balance').decode()}")
print(f"Bob's balance after transfer: {db.get(b'user:bob:balance').decode()}")

Alice's balance after transfer: 800
Bob's balance after transfer: 700


### Step 4: Transaction Rollback on Error
If an error occurs partway through a transaction, the partial writes are rolled back automatically.

In [10]:
# Simulate a failed transaction
try:
    with db.transaction() as txn:
        txn.put(b"user:alice:balance", b"0")  # Would set Alice to 0
        
        # Simulate an error before commit
        raise ValueError("Insufficient funds! Aborting transaction.")
        
except ValueError as e:
    print(f"Transaction aborted: {e}")

# Alice's balance should be unchanged (800 from previous transfer)
print(f"Alice's balance (unchanged): {db.get(b'user:alice:balance').decode()}")

Transaction aborted: Insufficient funds! Aborting transaction.
Alice's balance (unchanged): 800


### Step 5: Manual Transaction Control
If you need tighter control, you can manage the transaction lifecycle explicitly with transaction IDs.

In [11]:
txn = db.begin_transaction()

# Perform some writes against the transaction handle.
txn.put(b"user:carol:balance", b"750")
txn.put(b"user:dave:balance", b"250")

# Commit returns an HLC (Hybrid Logical Clock) timestamp for causal ordering.
hlc_timestamp = txn.commit()
print(f"Transaction committed at HLC timestamp: {hlc_timestamp}")

# Verify
print(f"Carol's balance: {db.get(b'user:carol:balance').decode()}")
print(f"Dave's balance: {db.get(b'user:dave:balance').decode()}")

Transaction committed at HLC timestamp: 4
Carol's balance: 750
Dave's balance: 250


### Step 6: Grouped Configuration Writes
A simple way to model related application state is to use consistent key prefixes.

In [12]:
config_items = [
    (b"config:max_retries", b"3"),
    (b"config:timeout_ms", b"5000"),
    (b"config:debug_mode", b"false"),
    (b"config:version", b"2.1.0"),
]

for key, value in config_items:
    db.put(key, value)

print(f"Inserted {len(config_items)} configuration keys.")

Inserted 4 configuration keys.


### Step 7: Prefix Scanning
Prefix scanning gives you a simple way to group and retrieve related keys such as config values, user records, or application state.

In [13]:
# Scan all keys starting with "config:"
config_entries = db.scan(b"config:")
print("All configuration entries:")
for entry in config_entries:
    print(f"  {entry}")

print()

# Scan all user balances
user_entries = db.scan(b"user:")
print("All user balance entries:")
for entry in user_entries:
    print(f"  {entry}")

All configuration entries:
  (b'config:debug_mode', b'false')
  (b'config:max_retries', b'3')
  (b'config:timeout_ms', b'5000')
  (b'config:version', b'2.1.0')

All user balance entries:
  (b'user:alice:balance', b'800')
  (b'user:bob:balance', b'700')
  (b'user:carol:balance', b'750')
  (b'user:dave:balance', b'250')


### Step 8: Hierarchical Key Design
You can mimic path-style organization with consistent key prefixes and then inspect related records with `scan`.

In [14]:
# Store data with hierarchical key prefixes
db.put(b"apps/assistant/config", json.dumps({"model": "local-default", "temperature": 0.7}).encode())
db.put(b"apps/assistant/prompts/system", b"You are a helpful assistant.")
db.put(b"apps/assistant/prompts/greeting", b"Hello! How can I help you today?")
db.put(b"apps/search/config", json.dumps({"index": "hybrid", "top_k": 10}).encode())

config = json.loads(db.get(b"apps/assistant/config").decode())
print(f"Assistant config: {config}")

print("\nAll keys under apps/assistant:")
for key, value in db.scan(b"apps/assistant"):
    print(f"  {key.decode()} → {value.decode()}")

Assistant config: {'model': 'local-default', 'temperature': 0.7}

All keys under apps/assistant:
  apps/assistant/config → {"model": "local-default", "temperature": 0.7}
  apps/assistant/prompts/greeting → Hello! How can I help you today?
  apps/assistant/prompts/system → You are a helpful assistant.


### Step 9: Inspect the Database Folder
Because SochDB is embedded, the database is just a local directory. That makes it easier to reason about durability and local state.

In [15]:
sorted(str(path.relative_to(DB_PATH)) for path in DB_PATH.rglob("*"))

['.lock', 'wal.log']

### Cleanup

Close the database cleanly when the walkthrough is done.

In [16]:
db.close()
print("Database closed.")

Database closed.
